# 1. Imports & Setup

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE

ModuleNotFoundError: No module named 'imblearn'

LOAD & SPLIT DATA

In [ ]:
# Split into Features (X) and Target (y)
X = df.drop(columns=['readmitted'])
y = df['readmitted']

In [4]:
# Split into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Initial Training Data Shape: {X_train.shape}")

NameError: name 'X' is not defined

# 3. PREPROCESSING (SCALING & PCA)

In [2]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ModuleNotFoundError: No module named 'imblearn'

Apply PCA to keep 95% of the variance

In [ ]:
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
print(f"Original dimensions before PCA: {X_train_scaled.shape[1]}")
print(f"Dimensions kept after PCA: {pca.n_components_}")

# 4. HANDLE CLASS IMBALANCE (SMOTE)

Balance the training data using SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_pca, y_train)

print(f"\nOriginal training target distribution:\n{y_train.value_counts()}")
print(f"Resampled training target distribution:\n{y_train_resampled.value_counts()}")

# 5. MODEL TRAINING

In [ ]:
from sklearn.ensemble import RandomForestClassifier

Initialize and train the Random Forest model

In [ ]:
# Initialize an optimized Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15, 
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1 # Uses all CPU cores
)

In [ ]:
# Train the model on the balanced data
print("\nTraining Random Forest Model (This might take a minute)...")
rf_model.fit(X_train_resampled, y_train_resampled)

In [ ]:
predictions = rf_model.predict(X_test_pca)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Print the overall accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"Overall Test Accuracy: {accuracy:.4f}\n")

# Print the detailed classification report
print("Classification Report:")
print(classification_report(y_test, predictions))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Generate the confusion matrix
cm = confusion_matrix(y_test, predictions)

# Plot it using Seaborn for a clean, professional look
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Class 0', 'Class 1', 'Class 2'], 
            yticklabels=['Class 0', 'Class 1', 'Class 2'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Random Forest Confusion Matrix')
plt.show()